# MinMaxScalerSklearn: Fit on Train, Transform Per Segment

This tutorial applies `MinMaxScalerSklearn` to `NamedTransformInput` data. The scaler uses `ConcatFitAndPerSegmentTransformMixin`: **fit on concatenated train**, **transform per segment**.

## Fit-on-Train Flow

```mermaid
flowchart TB
    subgraph Train["Train split"]
        T1[Chunk 1] --> Concat
        T2[Chunk 2] --> Concat
        T3[Chunk N] --> Concat
    end
    Concat[Concatenate] --> Fit[fit_multi_source]
    Fit --> Scaler[Scaler state: data_min_, data_max_]
    Scaler --> Transform[transform_data per chunk]
    
    subgraph ValTest["Val / Test splits"]
        V1[Chunk] --> Transform
    end
```

- **fit_multi_source(data_segments, metadata)** — Concatenates all train chunks, fits the underlying sklearn MinMaxScaler, stores min/max per feature.
- **transform_data(chunk, metadata)** — Applies the fitted scaler to each chunk. Val/test use train min/max; values can fall outside [0,1] if val/test have wider range.

## Config Metadata: apply_to, fit_on

When used in the pipeline (YAML config):

- **`apply_to`** — Which keys to read (e.g. `features`). Maps to `apply_to_keys` in metadata.
- **`fit_on`** — Which split to fit on (e.g. `train`). The pipeline calls `fit_multi_source` only on train chunks.

```yaml
scaler_features:
  transform:
    _target_: picid.transforms.base_transforms.scaler.MinMaxScalerSklearn
  metadata:
    apply_to: features
    fit_on: train
```

## Combine with Datasource Output

In a full pipeline, the datasource produces `SplitDatasetContainer` with train/val/test. The transform strategy builds chunks per split, fits on train chunks, then transforms all splits. This tutorial bypasses the pipeline and builds chunks manually to demonstrate the same API.

In [ ]:
import numpy as np
from picid.data.data_objects import NamedTransformInput
from picid.transforms.base_transforms.scaler import MinMaxScalerSklearn

metadata = {"apply_to_keys": ["features"]}

# Train chunks
arr1 = np.array([[1.0, 2.0], [3.0, 4.0]], dtype=np.float32)
arr2 = np.array([[5.0, 6.0], [7.0, 8.0]], dtype=np.float32)
train_chunks = [
    NamedTransformInput(features=arr1),
    NamedTransformInput(features=arr2),
]

# Val and test chunks
val_chunks = [
    NamedTransformInput(features=np.array([[2.0, 3.0], [4.0, 5.0]], dtype=np.float32)),
]
test_chunks = [
    NamedTransformInput(features=np.array([[0.5, 1.0], [9.0, 10.0]], dtype=np.float32)),
]

scaler = MinMaxScalerSklearn()
scaler.fit_multi_source(train_chunks, metadata)

assert scaler.scaler.data_min_ is not None, "Scaler should be fitted"
print(
    f"Fitted: data_min_={scaler.scaler.data_min_}, data_max_={scaler.scaler.data_max_}"
)

In [ ]:
# Transform train, val, test chunks
for i, chunk in enumerate(train_chunks):
    out = scaler.transform_data(chunk, metadata)
    assert out.shape == chunk["features"].shape
    print(f"Train chunk {i}: shape {out.shape}")

for i, chunk in enumerate(val_chunks):
    out = scaler.transform_data(chunk, metadata)
    assert out.shape == chunk["features"].shape
    print(f"Val chunk {i}: shape {out.shape}")

for i, chunk in enumerate(test_chunks):
    out = scaler.transform_data(chunk, metadata)
    assert out.shape == chunk["features"].shape
    print(f"Test chunk {i}: shape {out.shape}")

print("OK")

## Module Focus

This tutorial demonstrates **fit-on-train** scaling: fit once on concatenated train data, then transform each chunk (train, val, test) using the same learned min/max. For full pipeline integration with datasource output, see the transforms config and developer guides.